<a href="https://colab.research.google.com/github/tristanengst/FintechAndAICS983CourseStaff/blob/main/ColabNotebooks/ColabsFinGPT/3Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CASE3 Sentiment_Analysis_v3**

**Purpose**:

This notebook demonstrates how to load a LLaMA2-based FinGPT sentiment analysis model and generate sentiment predictions for financial news prompts.


---
## Instructions

**Step 1 – Switch to A100 runtime (Colab only)**  
> Purpose: loading FinGPT v3.2 (LLaMA2-based) requires a powerful GPU italicized text, such as A100 or RTX 3090.  

1. Go to **Runtime → Change runtime type** in the top menu.  
2. Set **Hardware accelerator** to `A100 GPU` or `L4` GPU.
3. To access a better GPU in Google Colab, you may need to:  
   - Upgrade to *Colab Pro*, or  
   - Use *Colab Pay-As-You-Go* (**recommended for flexibility**)  
   - [Click here](https://colab.research.google.com/signup) to upgrade or enable Pay-As-You-Go.  
4. Click **Save**.

**Step 2 – Run the entire notebook**  
Once the dependencies are installed and runtime is set, go to **Runtime → Run all** to execute all cells.

---
# worked for OS August 13

## Code

### Install Dependencies and Load FinGPT Code

---
⚠️ **Please switch to A100 runtime in Colab from this step**.

---

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 8.1 MB/s  0:00:05


In [ ]:
from google.colab import userdata

### FinGPT LLaMA2 Sentiment Prediction



In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast, BitsAndBytesConfig
from peft import PeftModel  # 0.5.0

# Load Models
base_model = "NousResearch/Llama-2-13b-hf"
peft_model = "FinGPT/fingpt-sentiment_llama2-13b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model = LlamaForCausalLM.from_pretrained(
    base_model,
    device_map="cuda:0",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
)  # force-set the model on a single GPU
# model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0", load_in_8bit = True,) # force-set the model on a single GPU
model = PeftModel.from_pretrained(model, peft_model)
model = model.eval()

# Make prompts
prompt = [
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is aggressively pursuing its growth strategy by increasingly focusing on technologically more demanding HDI printed circuit boards PCBs .
Answer: ''',
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
Answer: ''',
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: A tinyurl link takes users to a scamming site promising that users can earn thousands of dollars by becoming a Google ( NASDAQ : GOOG ) Cash advertiser .
Answer: ''',
]

# Generate results
tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512)
tokens = {k: v.to('cuda:0') for k, v in tokens.items()}

res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]

# show results
for sentiment in out_text:
    print(sentiment)

# Output:
# positive
# neutral
# negative

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/196 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

adapter_model.bin: reconstructing file:   0%|          |  0.00B / 19.7MB            

adapter_model.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2369: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


 positive</s>
 neutral</s>
 negative</s>


### Interactive Sentiment Analysis Loop

In [ ]:
# Loop to ask for user input
import re

while True:
    user_input = input("\nEnter a news text to analyze sentiment (or type 'exit' to quit): ")
    if user_input.strip().lower() == "exit":
        break

    # Build prompt
    prompt = f"""Instruction: What is the sentiment of this news? Please choose an answer from {{negative/neutral/positive}}
    Input: {user_input}
    Answer: """

    # Tokenize
    tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512, truncation=True)
    tokens = {k: v.to('cuda:0') for k, v in tokens.items()}

    res = model.generate(**tokens, max_length=512)
    res_sentences = [tokenizer.decode(i) for i in res]

    # Decode
    decoded = tokenizer.batch_decode(
        res,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    if "Answer:" in decoded[0]:
        after_answer = decoded[0].split("Answer:")[1].strip()
        match = re.search(r"[a-zA-Z\-]+", after_answer)
        if match:
            sentiment = match.group(0)
        else:
            sentiment = "Unable to parse output."
    else:
        sentiment = "Unable to parse output."

    print(f"\nPredicted Sentiment: {sentiment}")

# postive news example: Uncertainty isn't slowing Americans down; travel stocks are gaining as affluent consumers spend on premium experiences.
# negative news example: Fitch Ratings has kept Romania’s sovereign rating at BBB‑ (the lowest tier of investment grade) and assigned a negative outlook, citing deteriorating public finances, weak growth below 1%, fiscal policy implementation challenges, and mounting political polarization.


Enter a news text to analyze sentiment (or type 'exit' to quit): Uncertainty isn't slowing Americans down; travel stocks are gaining as affluent consumers spend on premium experiences.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Predicted Sentiment: positive


## Declaration

This section is adapted from the GitHub repository [`fingpt/FinGPT_Sentiment_Analysis_v3`](https://github.com/AI4Finance-Foundation/FinGPT/tree/master/fingpt/FinGPT_Sentiment_Analysis_v3).